# 🤖 CHRIST University Regex Chatbot

**CIA 1 — Advanced NLP | M.Tech Data Science**

---

## 1. Problem Statement & Objectives

### 1.1 Problem
Design and implement a **rule-based conversational chatbot** for CHRIST University that uses **Regular Expressions** as the primary NLP technique to understand user intent and generate meaningful responses.

### 1.2 Domain
**CHRIST University Information System** — covering admissions, online courses, hostel facilities, fees, accreditations, placements, library resources, student exchange programs, student council, and CICF.

### 1.3 Objectives
| # | Objective | How We Achieve It |
|---|-----------|-------------------|
| 1 | Handle diverse natural language queries | Advanced regex with lookaheads, named groups, optional params |
| 2 | Support multiple Indian languages | Malayalam, Tamil, Kannada via Unicode-aware patterns |
| 3 | Extract entities dynamically | Named capture groups (`?P<programme>`, `?P<gender>`, `?P<dept>`) |
| 4 | Generate context-aware responses | Template rendering, conditional answers, follow-up tracking |
| 5 | Graceful fallback for unmatched queries | IDF-weighted keyword matching + "Did you mean?" suggestions |
| 6 | Maintain conversational flow | Context manager resolves vague follow-ups using conversation history |

### 1.4 Why Regex over ML?
- **No training data required** — works out of the box with hand-crafted patterns
- **Fully interpretable** — every match can be explained and debugged
- **Lightweight** — no dependencies beyond Python's `re` module
- **Demonstrates NLP fundamentals** — tokenization, pattern matching, entity extraction

---
## 2. Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        USER INPUT                              │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────┐
│  PREPROCESSOR                                                    │
│  ├─ normalize_input()  → strip, collapse whitespace, NFC        │
│  ├─ detect_language()  → Unicode block analysis → en/ml/ta/kn   │
│  └─ classify_intent()  → keyword overlap → HOSTEL/ADMISSION/... │
└──────────────────────────┬───────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────┐
│  CONTEXT RESOLVER                                                │
│  └─ resolve_followup() → detects vague queries like "what about  │
│     the fees?" and prepends the last topic for richer matching   │
└──────────────────────────┬───────────────────────────────────────┘
                           │
              ┌────────────▼────────────┐
              │   REGEX MATCHER         │
              │   (Primary, 85–100%)    │
              │   Compiled patterns     │
              │   Longest match wins    │
              └────────────┬────────────┘
                           │ miss?
              ┌────────────▼────────────┐
              │  KEYWORD MATCHER        │
              │  (Fallback, 50–84%)     │
              │  IDF-weighted scoring   │
              │  Min 2 keyword matches  │
              └────────────┬────────────┘
                           │ miss?
              ┌────────────▼────────────┐
              │  SIMILAR QUESTIONS      │
              │  "Did you mean...?"     │
              │  Jaccard similarity     │
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  RESPONSE BUILDER       │
              │  ├─ Template + Data     │
              │  ├─ Conditional answers │
              │  ├─ Static fallback     │
              │  └─ Follow-up suggest.  │
              └─────────────────────────┘
```

### Module Structure
```
regex_chatbot/
├── engine/                      ← Core chatbot engine
│   ├── __init__.py              ← ChatEngine orchestrator class
│   ├── preprocessor.py          ← Input normalization, language detection, intent classification
│   ├── matcher.py               ← RegexMatcher + KeywordMatcher + similar questions
│   ├── response_builder.py      ← Dynamic response rendering, follow-ups, confidence labels
│   └── context.py               ← Conversation memory, follow-up resolution
├── qa_data/                     ← Q&A data (one JSON per team member)
│   ├── sanal_qa.json            ← IDs 1–17    (English, Malayalam)
│   ├── defitha_qa.json          ← IDs D1–D15  (English, Tamil)
│   ├── evengiline_qa.json       ← IDs E1–E15  (English, Kannada)
│   └── nasreen_qa.json          ← IDs N1–N15  (English, Hindi)
├── chatbot.py                   ← Streamlit Web App
├── regex_chatbot.ipynb          ← This notebook
├── requirements.txt             ← Dependencies
└── README.md                    ← Documentation
```

---
## 3. Environment Setup

In [1]:
import os
import sys

# Auto-detect if running on Google Colab
IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

QA_FOLDER = 'qa_data'

if IS_COLAB:
    print("☁️  Google Colab detected!")
    print("="*50)
    os.makedirs(QA_FOLDER, exist_ok=True)
    os.makedirs('engine', exist_ok=True)

    existing_files = [f for f in os.listdir(QA_FOLDER) if f.endswith('.json')]
    if existing_files:
        print(f"✅ Found {len(existing_files)} JSON file(s) already in qa_data/")
    else:
        from google.colab import files
        print("📤 Please upload your Q&A JSON files and engine/ module files:")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            if filename.endswith('.json'):
                filepath = os.path.join(QA_FOLDER, filename)
            elif filename.endswith('.py'):
                filepath = os.path.join('engine', filename)
            else:
                filepath = filename
            with open(filepath, 'wb') as f:
                f.write(content)
            print(f"✅ Saved '{filename}'")
else:
    if os.path.exists(QA_FOLDER):
        json_count = len([f for f in os.listdir(QA_FOLDER) if f.endswith('.json')])
        print(f"💻 Local environment detected.")
        print(f"✅ Found qa_data/ folder with {json_count} JSON file(s).")
    else:
        print("⚠️  qa_data/ folder not found!")

# Ensure engine module is importable
sys.path.insert(0, os.getcwd())
print(f"✅ Python path configured.")

💻 Local environment detected.
✅ Found qa_data/ folder with 4 JSON file(s).
✅ Python path configured.


In [2]:
import os
import sys

# Auto-detect if running on Google Colab
IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

QA_FOLDER = 'qa_data'

if IS_COLAB:
    print("☁️  Google Colab detected!")
    print("="*50)
    os.makedirs(QA_FOLDER, exist_ok=True)
    os.makedirs('engine', exist_ok=True)

    existing_files = [f for f in os.listdir(QA_FOLDER) if f.endswith('.json')]
    if existing_files:
        print(f"✅ Found {len(existing_files)} JSON file(s) already in qa_data/")
    else:
        from google.colab import files
        print("📤 Please upload your Q&A JSON files and engine/ module files:")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            if filename.endswith('.json'):
                filepath = os.path.join(QA_FOLDER, filename)
            elif filename.endswith('.py'):
                filepath = os.path.join('engine', filename)
            else:
                filepath = filename
            with open(filepath, 'wb') as f:
                f.write(content)
            print(f"✅ Saved '{filename}'")
else:
    if os.path.exists(QA_FOLDER):
        json_count = len([f for f in os.listdir(QA_FOLDER) if f.endswith('.json')])
        print(f"💻 Local environment detected.")
        print(f"✅ Found qa_data/ folder with {json_count} JSON file(s).")
    else:
        print("⚠️  qa_data/ folder not found!")

# Ensure engine module is importable
sys.path.insert(0, os.getcwd())
print(f"✅ Python path configured.")

💻 Local environment detected.
✅ Found qa_data/ folder with 4 JSON file(s).
✅ Python path configured.


---
## 4. Initialize the Chat Engine

In [3]:
from engine import ChatEngine
from engine.preprocessor import preprocess, detect_language, classify_intent
from engine.matcher import RegexMatcher, KeywordMatcher, find_similar_questions
from engine.response_builder import build_response, format_confidence_label, get_followup_suggestions
from engine.context import ConversationContext
import re
import json
import os

# Initialize the engine
engine = ChatEngine(QA_FOLDER)

print(f"✅ Loaded {len(engine.qa_data)} Q&A entries from {len(set(os.listdir(QA_FOLDER))) - 1} JSON files")
print(f"✅ {len(engine.regex_matcher.compiled_patterns)} regex patterns compiled successfully")

if engine.regex_matcher.errors:
    for err in engine.regex_matcher.errors:
        print(f"❌ {err}")
else:
    print("✅ Zero regex compilation errors")

# Show category distribution
from collections import Counter
categories = Counter(e.get('category', 'UNKNOWN') for e in engine.qa_data)
print(f"\n📊 Category Distribution:")
for cat, count in categories.most_common():
    print(f"   {cat}: {count} entries")

---
## 5. View All Loaded Questions

In [4]:
print("=" * 80)
print("📋 ALL LOADED QUESTIONS")
print("=" * 80)

for entry in engine.qa_data:
    qid = entry.get('id', '?')
    question = entry.get('question', 'N/A')
    question_en = entry.get('question_english', '')
    category = entry.get('category', 'GENERAL')
    has_template = '📐' if entry.get('answer_template') else ''
    has_conditional = '🔀' if entry.get('conditional_answers') else ''
    features = has_template + has_conditional

    if question_en:
        print(f"\n  [{category:12s}] {qid}. {question}")
        print(f"               (English: {question_en}) {features}")
    else:
        print(f"\n  [{category:12s}] {qid}. {question} {features}")

print("\n" + "=" * 80)
print("Legend: 📐 = Dynamic template response | 🔀 = Conditional answer variants")

---
## 6. Regex Pattern Analysis

This section demonstrates the **complexity and design** of our regex patterns. Each pattern is analyzed to show what regex features it uses.

In [5]:
print("=" * 100)
print("🔍 REGEX PATTERN ANALYSIS")
print("=" * 100)

def analyze_pattern(pattern_str):
    """Analyze a regex pattern and identify advanced features used."""
    features = []
    if '(?=' in pattern_str:
        features.append('Lookahead (?=)')
    if '(?P<' in pattern_str:
        # Extract group names
        groups = re.findall(r'\(\?P<(\w+)>', pattern_str)
        features.append(f'Named Groups: {groups}')
    if '(?:' in pattern_str:
        features.append('Non-capturing (?:)')
    if '\\s*' in pattern_str or '\\s+' in pattern_str:
        features.append('Whitespace flex')
    if '|' in pattern_str:
        alt_count = pattern_str.count('|')
        features.append(f'Alternation (|) ×{alt_count}')
    if '^' in pattern_str or '$' in pattern_str:
        features.append('Anchored (^/$)')
    if '[' in pattern_str:
        features.append('Character class []')
    if '?' in pattern_str and '(?:' not in pattern_str and '(?=' not in pattern_str and '(?P' not in pattern_str:
        features.append('Optional (?)')
    # Check for Unicode/multilingual
    if any(ord(c) > 127 for c in pattern_str):
        features.append('🌐 Multilingual')
    return features

for i, entry in enumerate(engine.qa_data):
    qid = entry.get('id', '?')
    question = entry.get('question_english', entry.get('question', ''))
    pattern = entry.get('pattern', '')
    features = analyze_pattern(pattern)

    if len(question) > 65:
        question = question[:62] + '...'

    print(f"\nQ{qid}: {question}")
    print(f"  Pattern: {pattern[:90]}{'...' if len(pattern) > 90 else ''}")
    print(f"  Features: {', '.join(features)}")

print("\n" + "=" * 100)

### 6.1 Regex Feature Examples

Here we demonstrate the key regex techniques used in our patterns:

| Technique | Pattern | What It Does | Example Match |
|-----------|---------|-------------|---------------|
| **Lookahead** | `(?=.*hostel)(?=.*fee).+` | Order-independent matching | "fee for hostel" AND "hostel fee" |
| **Named Groups** | `(?P<programme>bba\|bca)` | Captures entity for dynamic response | "BCA" from "fee for BCA" |
| **Non-capturing** | `(?:fees?\|cost)` | Groups alternatives without capturing | Matches "fee" or "fees" or "cost" |
| **Optional plural** | `courses?` | Handles singular and plural | "course" and "courses" |
| **Typo tolerance** | `accom[m]?od[ae]tion` | Handles common misspellings | "accommodation" and "accomodation" |
| **Anchored short** | `^hostel[s]?\\s*[?!.]*$` | Matches single-word queries exactly | Just "hostel" or "hostels" |
| **Unicode patterns** | `(ഹോസ്റ്റൽ\|താമസം)` | Matches Malayalam keywords | "ഹോസ്റ്റൽ സൗകര്യങ്ങൾ" |
| **Flexible whitespace** | `\\s+` / `\\s*` | Handles variable spacing | "student  exchange" |

---
## 7. Regex Accuracy Testing

We verify that **every sample query in our dataset actually matches its own parent pattern**. This proves our patterns are correct and comprehensive.

In [6]:
print("=" * 80)
print("🧪 REGEX ACCURACY TEST")
print("    Testing: Does each sample_query match its parent entry's pattern?")
print("=" * 80)

total = 0
correct = 0
misses = []

for entry in engine.qa_data:
    pattern_str = entry.get('pattern', '')
    qid = entry.get('id', '?')
    try:
        compiled = re.compile(pattern_str, re.IGNORECASE)
    except re.error:
        continue

    for query in entry.get('sample_queries', []):
        total += 1
        if compiled.search(query):
            correct += 1
        else:
            misses.append((qid, query, pattern_str[:60]))

accuracy = (correct / total * 100) if total > 0 else 0

print(f"\n📊 Results: {correct}/{total} sample queries matched their parent pattern")
print(f"📊 Accuracy: {accuracy:.1f}%")

if misses:
    print(f"\n⚠️  {len(misses)} misses (queries that didn't match their own pattern):")
    for qid, query, pat in misses:
        print(f"   Q{qid}: '{query}'")
        print(f"      Pattern: {pat}...")
else:
    print("\n✅ All sample queries match their parent patterns — zero misses!")

print("\n" + "=" * 80)

### 7.1 Cross-Match Test

We also verify that sample queries **don't accidentally match a different entry better** than their intended one.

In [7]:
print("=" * 80)
print("🔄 CROSS-MATCH TEST")
print("    Testing: Does the engine return the CORRECT entry for each sample query?")
print("=" * 80)

total = 0
correct = 0
wrong = []

for entry in engine.qa_data:
    qid = str(entry.get('id', '?'))
    for query in entry.get('sample_queries', []):
        total += 1
        # Use a fresh engine to avoid context effects
        result = engine.regex_matcher.match(query)
        if result:
            matched_entry, _, _ = result
            matched_id = str(matched_entry.get('id', ''))
            if matched_id == qid:
                correct += 1
            else:
                wrong.append((qid, matched_id, query[:50]))
        else:
            # No regex match — check keyword fallback
            kw_result = engine.keyword_matcher.match(query)
            if kw_result:
                matched_entry, _ = kw_result
                matched_id = str(matched_entry.get('id', ''))
                if matched_id == qid:
                    correct += 1
                else:
                    wrong.append((qid, matched_id, query[:50]))
            else:
                wrong.append((qid, 'NONE', query[:50]))

accuracy = (correct / total * 100) if total > 0 else 0
print(f"\n📊 Results: {correct}/{total} queries matched the CORRECT entry")
print(f"📊 Cross-Match Accuracy: {accuracy:.1f}%")

if wrong:
    print(f"\n⚠️  {len(wrong)} incorrect matches:")
    for expected, got, query in wrong[:10]:
        print(f"   Expected Q{expected}, got Q{got}: '{query}'")
else:
    print("\n✅ All queries route to the correct entry — zero cross-matches!")

print("\n" + "=" * 80)

---
## 8. Dynamic Response Demonstration

This section demonstrates the **three dynamic response strategies** our chatbot uses:
1. **Template + Data** — entity extraction fills a template with specific data
2. **Conditional Answers** — different answers based on detected sub-entities
3. **Static Fallback** — comprehensive static answer when no entity is detected

In [8]:
print("=" * 80)
print("📐 DYNAMIC RESPONSE DEMO — Template + Entity Extraction")
print("=" * 80)

# --- Strategy 1: Template + Data (fee queries from sanal_qa) ---
fee_queries = [
    "What is the fee for BBA?",
    "BCA fees and duration",
    "How much does MCA cost?",
    "Tell me the fees and duration",  # No specific programme → static fallback
]

print("--- Strategy 1: Template + Data (fee queries — sanal_qa) ---")
for q in fee_queries:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(q)
    print(f"\n  👤 {q}")
    print(f"  🤖 {r['response'][:120]}{'...' if len(r['response']) > 120 else ''}")
    print(f"     [{r['confidence_label']}]")

print("\n")
print("--- Strategy 2: Conditional Answers (gender-specific hostel — sanal_qa) ---")
hostel_queries = [
    "boys hostel fees",          # Boys → conditional answer (ID 7)
    "girls hostel fee",          # Girls → conditional answer (ID 7)
    "Is hostel available for girls?",  # Girls → conditional answer (ID 4)
    "What is the hostel admission process?",  # No gender → ID 6 (admission process)
]

for q in hostel_queries:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(q)
    print(f"\n  👤 {q}")
    print(f"  🤖 {r['response'][:120]}{'...' if len(r['response']) > 120 else ''}")
    print(f"     [{r['confidence_label']}]")

print("\n")
print("--- Strategy 3: Conditional Answers (department-specific exchange — sanal_qa) ---")
exchange_queries = [
    "engineering exchange program",
    "law exchange",
    "psychology exchange opportunities",
    "Where can students go for exchange?",  # No dept → full list
]

for q in exchange_queries:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(q)
    print(f"\n  👤 {q}")
    print(f"  🤖 {r['response'][:120]}{'...' if len(r['response']) > 120 else ''}")
    print(f"     [{r['confidence_label']}]")

print("\n")
print("--- Strategy 4: Static Responses from Other Team Members ---")
cross_team_queries = [
    # defitha_qa (Examination, Placement)
    "When will the exam timetable be released?",
    "How to register for placements?",
    "What is IQAC?",
    # evengiline_qa (Admission, Library, CICF)
    "Is there an entrance test for admission?",
    "How do I access library resources remotely?",
    "What services does CICF offer?",
    # nasreen_qa (SDG, PhD, Payment, Alumni)
    "What activities were conducted by the SDG Cell?",
    "What is the process for PhD admission?",
    "Is refund available after payment?",
    "How does alumni network help students?",
]

for q in cross_team_queries:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(q)
    print(f"\n  👤 {q}")
    print(f"  🤖 {r['response'][:120]}{'...' if len(r['response']) > 120 else ''}")
    print(f"     [{r['confidence_label']}]")

print("\n" + "=" * 80)


---
## 9. Multilingual Support Demonstration

Our chatbot supports queries in **4 languages**: English, Malayalam, Tamil, and Kannada. The language is auto-detected via Unicode block analysis.

In [9]:
print("=" * 80)
print("🌐 MULTILINGUAL SUPPORT DEMO")
print("=" * 80)

LANG_NAMES = {'en': 'English', 'ml': 'Malayalam', 'ta': 'Tamil', 'kn': 'Kannada', 'hi': 'Hindi'}

multilingual_queries = [
    # English (sanal_qa)
    "What are the hostel facilities?",
    # Malayalam — Online Courses (sanal_qa ID 1)
    "ക്രൈസ്റ്റ് ഓൺലൈൻ പ്രോഗ്രാമുകൾ എന്തൊക്കെയാണ്?",
    # Malayalam — Accreditations (sanal_qa ID 10, newly bilingual)
    "ക്രൈസ്റ്റ് യൂണിവേഴ്സിറ്റിയുടെ NAAC ഗ്രേഡ് എന്താണ്?",
    # Malayalam — Exchange Overview (sanal_qa ID 11, newly bilingual)
    "ക്രൈസ്റ്റ് യൂണിവേഴ്സിറ്റിയിൽ എക്സ്ചേഞ്ച് പ്രോഗ്രാം ഉണ്ടോ?",
    # Malayalam — Central Campus Girls Hostel (sanal_qa ID 8)
    "സെന്റ്രൽ ക്യാമ്പസിന് സമീപമുള്ള പെൺകുട്ടികളുടെ ഹോസ്റ്റലുകൾ ഏതൊക്കെയാണ്?",
    # Malayalam — Central Campus Boys Hostel (sanal_qa ID 9)
    "സെന്റ്രൽ ക്യാമ്പസിന് സമീപമുള്ള ആൺകുട്ടികളുടെ ഹോസ്റ്റലുകൾ ഏതൊക്കെയാണ്?",
    # Tamil (defitha_qa D11)
    "மதிப்பெண் பட்டியல் எப்போது வெளியிடப்படும்?",
    # Kannada (evengiline_qa E1)
    "ಕ್ರೈಸ್ಟ್ ವಿಶ್ವವಿದ್ಯಾಲಯಕ್ಕೆ ನಾನು ಹೇಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಬೇಕು?",
    # Hindi (nasreen_qa N11)
    "SDG सेल क्या करता है?",
]

for q in multilingual_queries:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(q)
    lang = LANG_NAMES.get(r['language'], r['language'])
    print(f"\n  🗣️  Language: {lang}")
    print(f"  👤 {q[:80]}{'...' if len(q) > 80 else ''}")
    print(f"  🤖 {r['response'][:100]}{'...' if len(r['response']) > 100 else ''}")
    print(f"     [{r['confidence_label']}] [Intent: {r['intent']}]")

print("\n" + "=" * 80)


---
## 10. Conversational Flow & Context Demo

This demonstrates how the chatbot maintains **conversation context** across turns, resolves vague follow-ups, and suggests related questions.

In [10]:
print("=" * 80)
print("💬 CONVERSATION FLOW DEMO")
print("=" * 80)

# Simulate a multi-turn conversation
e = ChatEngine(QA_FOLDER)
conversation = [
    "hi",
    "Tell me about the hostel",
    "What about the fees?",         # Follow-up → should infer "hostel fees"
    "boys hostel fees",              # Specific conditional
    "What is the fee for BCA?",      # New topic with entity extraction
    "Is CHRIST accredited?",         # Topic switch
    "thanks",
    "bye",
]

for q in conversation:
    r = e.respond(q)
    short = r['response'][:120] + '...' if len(r['response']) > 120 else r['response']
    print(f"\n  👤 You: {q}")
    print(f"  🤖 Bot: {short}")
    print(f"     [{r['confidence_label']}] [Intent: {r['intent']}]")
    if r.get('followups'):
        print(f"     💡 Follow-ups: {r['followups'][:2]}")

print("\n" + "=" * 80)

---
## 11. Full Test Suite

Testing across all query types: full questions, short keywords, multilingual, edge cases, and out-of-scope queries.

In [11]:
print("=" * 80)
print("🧪 FULL TEST SUITE")
print("=" * 80)

test_cases = [
    # ========== sanal_qa.json (IDs 1–17) ==========
    # Academics & Admission
    ("What online courses are offered?", "regex"),
    ("How can I apply for CHRIST Online?", "regex"),
    ("What is the fee for BCA?", "regex"),                    # Template response
    # Hostel (split: admission process vs fees)
    ("Is hostel available in CHRIST University?", "regex"),
    ("What are the hostel amenities?", "regex"),
    ("What is the hostel admission process?", "regex"),       # ID 6 (split)
    ("What are the hostel fees?", "regex"),                   # ID 7 (split)
    ("boys hostel fees", "regex"),                            # Conditional answer
    ("girls hostel near central campus", "regex"),             # ID 8 (split)
    ("boys hostel near central campus", "regex"),              # ID 9 (split)
    # Accreditations & Exchange (newly bilingual)
    ("Is CHRIST University NAAC accredited?", "regex"),       # ID 10
    ("Tell me about the student exchange program.", "regex"), # ID 11
    ("engineering exchange program", "regex"),                # ID 13 conditional
    # Exchange splits
    ("How can I apply for outgoing exchange?", "regex"),      # ID 14 (split)
    ("Am I eligible for the student exchange program?", "regex"), # ID 15 (split)

    # ========== defitha_qa.json (IDs D1–D15) ==========
    ("When will the exam timetable be released?", "regex"),   # D1
    ("Where can I find previous question papers?", "regex"),  # D2
    ("How to register for placements?", "regex"),             # D7
    ("What rounds are included in campus recruitment?", "regex"), # D8
    ("What is IQAC?", "regex"),                               # D10

    # ========== evengiline_qa.json (IDs E1–E15) ==========
    ("What programmes does Christ University offer?", "regex"), # E2
    ("Is there an entrance test for admission?", "regex"),    # E3
    ("What is Daksh?", "regex"),                              # E6
    ("What services does CICF offer?", "regex"),              # E10
    ("How do I access library resources remotely?", "regex"), # E14

    # ========== nasreen_qa.json (IDs N1–N15) ==========
    ("What activities were conducted by the SDG Cell?", "regex"), # N1
    ("What is the process for PhD admission?", "regex"),      # N3
    ("How can I make payment online?", "regex"),              # N6
    ("Is refund available after payment?", "regex"),          # N8
    ("How does alumni network help students?", "regex"),      # N10

    # ========== Short keyword queries ==========
    ("hostel", "regex"),
    ("NAAC", "regex"),
    ("CICF", "regex"),
    ("placement", "regex"),

    # ========== Multilingual queries ==========
    ("പെൺകുട്ടികളുടെ ഹോസ്റ്റൽ", "regex"),                  # Malayalam
    ("ആൺകുട്ടികളുടെ ഹോസ്റ്റൽ", "regex"),                   # Malayalam
    ("மதிப்பெண் பட்டியல் எப்போது வெளியிடப்படும்?", "regex"), # Tamil
    ("SDG सेल क्या करता है?", "regex"),                       # Hindi

    # ========== Out-of-scope (should NOT match) ==========
    ("What is the weather in Bangalore?", "none"),
    ("Who is the prime minister?", "none"),
]

passed = 0
failed = 0

for query, expected_type in test_cases:
    e = ChatEngine(QA_FOLDER)
    r = e.respond(query)
    actual_type = r['match_type']

    # For expected 'regex', also accept 'keyword' as partial success
    if expected_type == 'regex':
        ok = actual_type in ('regex', 'keyword')
    elif expected_type == 'none':
        ok = actual_type == 'none'
    else:
        ok = actual_type == expected_type

    status = '✅' if ok else '❌'
    if ok:
        passed += 1
    else:
        failed += 1

    short_resp = r['response'][:60] + '...' if len(r['response']) > 60 else r['response']
    print(f"  {status} [{actual_type:7s}] {query[:50]:50s} → {short_resp}")

print(f"\n📊 Test Results: {passed}/{passed+failed} passed ({passed/(passed+failed)*100:.0f}%)")
print("=" * 80)


---
## 12. Interactive Chat 💬

Type your questions to chat with the bot! Features shown:
- **Confidence score** for every answer
- **Intent classification** and **language detection**
- **Follow-up suggestions** after each answer
- **Context-aware follow-ups** (vague queries use previous topic)

Type `quit` or `exit` to end.

In [12]:
LANG_NAMES = {'en': 'English', 'ml': 'Malayalam', 'ta': 'Tamil', 'kn': 'Kannada', 'hi': 'Hindi'}

print("=" * 70)
print("  🤖 CHRIST UNIVERSITY CHATBOT")
print("  Powered by Regular Expressions | Dynamic Entity Extraction")
print("=" * 70)
print()
print("  Welcome! Ask me anything about CHRIST University.")
print("  Type 'help' to see topics | Type 'quit' to exit")
print()
print("-" * 70)

chat_engine = ChatEngine(QA_FOLDER)

while True:
    try:
        user_input = input("\n👤 You: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n\n🤖 Bot: Goodbye! 👋 Have a great day!")
        break

    if not user_input:
        continue

    if user_input.lower() in ['quit', 'exit', 'bye', 'stop']:
        result = chat_engine.respond(user_input)
        print(f"\n🤖 Bot: {result['response']}")
        print("\n" + "=" * 70)
        break

    result = chat_engine.respond(user_input)

    lang = LANG_NAMES.get(result['language'], result['language'])

    print(f"\n🤖 Bot: {result['response']}")
    print(f"\n   📊 {result['confidence_label']} | 🏷️ {result['intent']} | 🗣️ {lang}")

    if result.get('followups'):
        print(f"\n   💡 You might also want to know:")
        for fq in result['followups'][:3]:
            print(f"      • {fq}")

    print("\n" + "-" * 70)

---
## 13. Summary & Conclusions

### What We Built
A **modular, rule-based chatbot** that demonstrates advanced Regular Expression techniques for natural language understanding.

### Key Technical Achievements

| Feature | Implementation |
|---------|---------------|
| **Advanced Regex** | Lookaheads, named groups, non-capturing groups, optional params, anchors, character classes |
| **Dynamic Responses** | Template rendering with entity extraction, conditional answer variants |
| **Multilingual** | Malayalam, Tamil, Kannada, Hindi support via Unicode-aware patterns |
| **Two-level Matching** | Primary regex (85-100% confidence) + IDF-weighted keyword fallback (50-84%) |
| **Conversation Context** | Follow-up resolution, repeated question detection, session summary |
| **Graceful Fallback** | "Did you mean?" suggestions using Jaccard similarity |
| **Modular Architecture** | 5-module engine: preprocessor → matcher → response_builder → context → orchestrator |

### Limitations
- Regex cannot handle truly open-ended NLU (e.g., paraphrasing, sarcasm, complex reasoning)
- New topics require manual pattern creation — no automatic learning
- Keyword fallback can produce false positives for very short queries

### Future Improvements
- Add spell correction preprocessing (Levenshtein distance)
- Integrate with a vector similarity model for semantic fallback
- Add voice input support
- Connect to live university databases for real-time data

---

### Team Members

| Name       | File                    | IDs    | Languages          |
|------------|------------------------|--------|--------------------|
| Sanal      | `sanal_qa.json`        | 1–17   | English, Malayalam |
| Defitha    | `defitha_qa.json`      | D1–D15 | English, Tamil     |
| Evengiline | `evengiline_qa.json`   | E1–E15 | English, Kannada   |
| Nasreen    | `nasreen_qa.json`      | N1–N15 | English, Hindi     |
